# 1. Reading CSV from ADLS Gen2 in Databricks using SAS

In [0]:
# 1. Reading CSV from ADLS Gen2 in Databricks using SAS

storage_account_name = "hexadbwb"
container_name = "azurecs"
sas_token = "sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2025-08-18T14:33:12Z&st=2025-08-18T06:18:12Z&spr=https&sig=RQvdsYzb9GGZUS%2BCxukySx2CUE%2BsUrdj%2FyPOFyacOxM%3D"
file_name = "Employee.csv"


# Path in wasbs format
file_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/{file_name}"

# Set Spark config with SAS token
spark.conf.set(f"fs.azure.sas.{container_name}.{storage_account_name}.blob.core.windows.net", sas_token)

# Read the CSV
df = (spark.read
      .format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load(file_path))

display(df.limit(5))

Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
Bachelors,2017,Bangalore,3,34,Male,No,0,0
Bachelors,2013,Pune,1,28,Female,No,3,1
Bachelors,2014,New Delhi,3,38,Female,No,2,0
Masters,2016,Bangalore,3,27,Male,No,5,1
Masters,2017,Pune,3,24,Male,Yes,2,1


# 2. Data Overview

In [0]:
print("Schema:")
df.printSchema()

print("\nSummary Statistics:")
display(df.describe())

print("\nTotal Records:", df.count())


Schema:
root
 |-- Education: string (nullable = true)
 |-- JoiningYear: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: string (nullable = true)
 |-- ExperienceInCurrentDomain: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)


Summary Statistics:


summary,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
count,4653,4653,4653,4653,4653,4653,4653,4653,4653
mean,null,2015.0629701267999,null,2.6982591876208897,29.393294648613796,null,null,2.905652267354395,0.3438641736514077
stddev,null,1.863376828686355,null,0.5614354643364909,4.826087009126064,null,null,1.5582403309268569,0.47504747514881024
min,Bachelors,2012,Bangalore,1,22,Female,No,0,0
max,PHD,2018,Pune,3,41,Male,Yes,7,1



Total Records: 4653


# 3. Save as Delta Table

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("EmployeesDelta")

print("Delta table created:")
display(spark.sql("SELECT * FROM EmployeesDelta LIMIT 5"))

Delta table created:


Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
Bachelors,2017,Bangalore,3,34,Male,No,0,0
Bachelors,2013,Pune,1,28,Female,No,3,1
Bachelors,2014,New Delhi,3,38,Female,No,2,0
Masters,2016,Bangalore,3,27,Male,No,5,1
Masters,2017,Pune,3,24,Male,Yes,2,1


# 4. Transformations
# 4.1 Select specific columns

In [0]:
display(
  df.select(
    "Age",
    "City",
    "Education",
    "Gender",
    "EverBenched"
  ).limit(5)
)

Age,City,Education,Gender,EverBenched
34,Bangalore,Bachelors,Male,No
28,Pune,Bachelors,Female,No
38,New Delhi,Bachelors,Female,No
27,Bangalore,Masters,Male,No
24,Pune,Masters,Male,Yes


# 4.2 Filter records (example: Age > 30)

In [0]:
df.filter(df["Age"] > 30).show(5)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Bachelors|       2017|Bangalore|          3| 34|  Male|         No|                        0|         0|
|Bachelors|       2014|New Delhi|          3| 38|Female|         No|                        2|         0|
|Bachelors|       2015|New Delhi|          3| 38|  Male|         No|                        0|         0|
|Bachelors|       2016|Bangalore|          3| 34|Female|         No|                        2|         1|
|  Masters|       2017|New Delhi|          2| 37|  Male|         No|                        2|         0|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
only showing top 5 rows


# 4.3 Group by Education and average Age

In [0]:
from pyspark.sql.functions import avg

display(
  df.groupBy("Education").agg(
    avg("Age").alias("Average_Age")
  )
)

Education,Average_Age
Masters,29.29095074455899
Bachelors,29.422938072757567
PHD,29.29608938547486


# 4.4 Order by Age descending

In [0]:
df.orderBy(df["Age"].desc()).show(5)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|  Masters|       2017|Bangalore|          3| 41|Female|         No|                        2|         1|
|Bachelors|       2013|New Delhi|          3| 41|Female|         No|                        4|         0|
|Bachelors|       2015|     Pune|          3| 41|  Male|         No|                        5|         0|
|Bachelors|       2016|Bangalore|          3| 41|  Male|         No|                        0|         0|
|Bachelors|       2017|Bangalore|          3| 41|  Male|        Yes|                        3|         0|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
only showing top 5 rows


# 5. Visualizations

# 5.1 Bar Chart - Average Age by Education

In [0]:
# 5.1 Bar Chart - Average Age by Education
print("\n=== Visualization: Average Age by Education ===")

from pyspark.sql.functions import avg

# Step 1: Group and aggregate
avg_age_by_edu = df.groupBy("Education").agg(avg("Age").alias("Average_Age"))

# Step 2: Display as bar chart
display(avg_age_by_edu)



=== Visualization: Average Age by Education ===


Education,Average_Age
Masters,29.29095074455899
Bachelors,29.422938072757567
PHD,29.29608938547486



# 5.2 Pie Chart - Gender Distribution

In [0]:
from pyspark.sql.functions import count
display(df.groupBy("Gender").agg(count("*").alias("Count")))


Gender,Count
Female,1875
Male,2778
